In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
feature_names = iris.feature_names

df = pd.DataFrame(X, columns=feature_names)
df.head()

In [ ]:
print('Number of records:', df.shape[0])
print('Number of columns:', df.shape[1])

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), dpi=80)
df.hist(ax=ax, layout=(2, 2), alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df)

In [ ]:
sns.heatmap(df.corr(numeric_only=True), cmap='RdYlBu', annot=True)
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

def normalize(X):
    print("Mean and Standard Deviation Before")
    print(X.mean(axis=0), X.std(axis=0))

    sc = StandardScaler()
    XScaled = sc.fit_transform(X)

    print("Mean and Standard Deviation After")
    print(XScaled.mean(axis=0).round(4), XScaled.std(axis=0))
    return XScaled

In [ ]:
print("*************Normalization/Standardization*************")
XScaled = normalize(X)

In [ ]:
from sklearn.decomposition import PCA

def applyPCA(XScaled, n_components=2):
    print("*************Applying PCA*************")
    print("Shape Before PCA:", XScaled.shape)

    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(XScaled)

    print("Shape After PCA:", X_pca.shape)
    print("Explained Variance Ratio:", pca.explained_variance_ratio_)
    print("Total Variance Retained:", sum(pca.explained_variance_ratio_).round(4))
    return X_pca

In [ ]:
X_pca = applyPCA(XScaled, n_components=2)

In [ ]:
from sklearn.cluster import KMeans

# Elbow Method to find optimal K
print("*************Elbow Method for Optimal K*************")
inertia = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o', color='steelblue')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (WCSS)')
plt.title('Elbow Method for Optimal K')
plt.xticks(k_range)
plt.grid(True)
plt.show()

In [ ]:
def KMeansClustering(X_pca, n_clusters=3):
    print("*************K-Means Clustering (K =", n_clusters, ")*************")

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_pca)

    print("Cluster Centers (in PCA space):")
    print(kmeans.cluster_centers_)
    print("Inertia (WCSS):", kmeans.inertia_)

    print("\n*************Visualizing K-Means Clusters*************")
    plt.figure(figsize=(8, 6))
    colors = ['red', 'green', 'blue', 'orange', 'purple']
    for i in range(n_clusters):
        plt.scatter(X_pca[labels == i, 0], X_pca[labels == i, 1],
                    c=colors[i], label=f'Cluster {i+1}', alpha=0.7, edgecolors='k', s=60)
    plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                c='black', marker='X', s=200, label='Centroids')
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.title(f'K-Means Clustering (K={n_clusters})')
    plt.legend()
    plt.grid(True)
    plt.show()

    return labels

In [ ]:
kmeans_labels = KMeansClustering(X_pca, n_clusters=3)

In [ ]:
from sklearn.metrics import silhouette_score

print("*************Silhouette Score for K-Means*************")
score = silhouette_score(X_pca, kmeans_labels)
print("Silhouette Score (K=3):", round(score, 4))

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# Plot Dendrogram to decide number of clusters
print("*************Dendrogram for Agglomerative Clustering*************")
linked = linkage(X_pca, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(linked,
           truncate_mode='lastp',
           p=30,
           leaf_rotation=90,
           leaf_font_size=10,
           show_contracted=True)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Sample Index / Cluster Size')
plt.ylabel('Ward Distance')
plt.grid(True)
plt.show()

In [ ]:
def AgglomerativeClusteringModel(X_pca, n_clusters=3, linkage_method='ward'):
    print("*************Agglomerative Clustering (n_clusters =", n_clusters, ", linkage =", linkage_method, ")*************")

    agglo = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage_method)
    labels = agglo.fit_predict(X_pca)

    print("Unique Cluster Labels:", np.unique(labels))
    print("Cluster Sizes:", {i: np.sum(labels == i) for i in np.unique(labels)})

    print("\n*************Visualizing Agglomerative Clusters*************")
    plt.figure(figsize=(8, 6))
    colors = ['red', 'green', 'blue', 'orange', 'purple']
    for i in range(n_clusters):
        plt.scatter(X_pca[labels == i, 0], X_pca[labels == i, 1],
                    c=colors[i], label=f'Cluster {i+1}', alpha=0.7, edgecolors='k', s=60)
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.title(f'Agglomerative Clustering (n={n_clusters}, linkage={linkage_method})')
    plt.legend()
    plt.grid(True)
    plt.show()

    return labels

In [ ]:
agglo_labels = AgglomerativeClusteringModel(X_pca, n_clusters=3, linkage_method='ward')

In [ ]:
print("*************Silhouette Score for Agglomerative Clustering*************")
score_agglo = silhouette_score(X_pca, agglo_labels)
print("Silhouette Score (n=3, ward):", round(score_agglo, 4))

In [ ]:
print("*************Comparison: K-Means vs Agglomerative Clustering*************")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['red', 'green', 'blue']

# K-Means
for i in range(3):
    axes[0].scatter(X_pca[kmeans_labels == i, 0], X_pca[kmeans_labels == i, 1],
                    c=colors[i], label=f'Cluster {i+1}', alpha=0.7, edgecolors='k', s=60)
axes[0].set_title('K-Means Clustering (K=3)')
axes[0].set_xlabel('PCA Component 1')
axes[0].set_ylabel('PCA Component 2')
axes[0].legend()
axes[0].grid(True)

# Agglomerative
for i in range(3):
    axes[1].scatter(X_pca[agglo_labels == i, 0], X_pca[agglo_labels == i, 1],
                    c=colors[i], label=f'Cluster {i+1}', alpha=0.7, edgecolors='k', s=60)
axes[1].set_title('Agglomerative Clustering (ward, n=3)')
axes[1].set_xlabel('PCA Component 1')
axes[1].set_ylabel('PCA Component 2')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('K-Means vs Agglomerative Clustering on Iris (PCA Reduced)', fontsize=13)
plt.tight_layout()
plt.show()

print(f"K-Means Silhouette Score     : {round(silhouette_score(X_pca, kmeans_labels), 4)}")
print(f"Agglomerative Silhouette Score: {round(silhouette_score(X_pca, agglo_labels), 4)}")